# SQL Practice Notebook (Databricks)

In this notebook, we will cover:

- Creating tables
- Querying data
- Filtering, sorting
- Aggregations
- Joins
- Window functions
- CTEs and subqueries
- Writing data

All examples follow real-world data engineering scenarios.

## Step 1: Create Raw Table (Bronze Layer)

We simulate raw ingested data.

In [0]:
create or replace table practice.bronze.orders_raw (
  order_id INT,
  customer_id INT,
  product_id INT,
  order_date STRING,
  quantity INT,
  price DOUBLE,
  status string
);

In [0]:
INSERT into practice.bronze.orders_raw  (order_id, customer_id, product_id, order_date, quantity, price, status)
values
(1, 101, 1001, '2024-01-01', 2, 500, 'completed'),
(2, 102, 1002, '2024-01-02', 1, 300, 'pending'),
(3, 103, 1003, '2024-01-03', 5, 200, 'completed'),
(4, 104, 1004, '2024-01-04', 1, 500, 'cancelled'),
(5, 105, 1005, '2024-01-05', 3, 300, 'completed'),
(6, 106, 1006, '2024-01-06', 2, 700, 'completed'),
(7, 107, 1007, '2024-01-07', 1, 500, 'pending');



## Step 2: Basic SELECT Queries

In [0]:
select * from practice.bronze.orders_raw;

In [0]:
select order_id, customer_id, price from practice.bronze.orders_raw;

In [0]:
select distinct customer_id from practice.bronze.orders_raw;

In [0]:
select * from practice.bronze.orders_raw
LIMIT 5;

## Step 3: Filtering Data

In [0]:
select * from practice.bronze.orders_raw
where status = 'completed';

In [0]:
select * from practice.bronze.orders_raw
where price > 400;

## Step 4: Sorting Results

In [0]:
select * from practice.bronze.orders_raw
where quantity >=2;

In [0]:
select * from practice.bronze.orders_raw
where price > 300 AND status = 'completed';

In [0]:
select * from practice.bronze.orders_raw
where customer_id NOT IN (101,102);

## Step 5: Aggregations
(It means summarizing data)

*Basic Syntax* = select column, AGG_FUN(column) from table GROUP BY column;

In [0]:
-- TOTAL REVENUE

select SUM(quantity*price) AS total_revenue
from practice.bronze.orders_raw;

In [0]:
select COUNT(*) AS total_orders
from practice.bronze.orders_raw;

In [0]:
select customer_id, AVG(price) from practice.bronze.orders_raw
GROUP BY customer_id;

In [0]:
select customer_id, sum(price*quantity) as total_spent
from practice.bronze.orders_raw
GROUP BY customer_id;

In [0]:
-- Orders per status

select status, COUNT(*) as total_orders
from practice.bronze.orders_raw
GROUP BY status;

In [0]:
-- MAX and MIN price
select 
  MAX(price) AS max_price,
  MIN(price) AS min_price
from practice.bronze.orders_raw;

## Step 6: HAVING Clause
(used to filter after Aggregation)

* WHERE = filter before grouping
* HAVING = after Grouping

*BASIC SYNTAX* =
SELECT column, AGG_FUN(column)
FROM TABLE
GROUP BY column
HAVING condition

In [0]:
-- CUSt spending > 1000

select customer_id, SUM(price*quantity) as total_spent
from practice.bronze.orders_raw
GROUP BY customer_id
HAVING total_spent > 1000;

In [0]:
-- Status with more than two orders

select status , COUNT(*) as total_orders
from practice.bronze.orders_raw
GROUP BY status
HAVING total_orders >= 2;


In [0]:
select product_id, SUM(price*quantity) as revenue
from practice.bronze.orders_raw
GROUP BY product_id
having revenue > 500;

In [0]:
select customer_id, avg(price) as avg_price
from practice.bronze.orders_raw
group by customer_id
HAVING avg_price > 300;

In [0]:
-- Product sold more than three times
select product_id, COUNT(*) as total_sold
from practice.bronze.orders_raw
GROUP BY product_id
HAVING total_sold > 3

In [0]:
select 
  to_date(order_date) as order_day,
  COUNT(*) as total_orders
from practice.bronze.orders_raw
GROUP BY to_date(order_date)
having total_orders > 2;

## Step 7: Create Customer Table (Dimension)

In [0]:
CREATE OR REPLACE TABLE practice.bronze.dim_customers (
customer_id INT,
customer_name STRING,
city STRING,
state STRING,
signup_date DATE
);

INSERT INTO practice.bronze.dim_customers VALUES
(101, 'Ravi Kumar', 'Delhi', 'Delhi', '2023-01-10'),
(102, 'Amit Sharma', 'Mumbai', 'Maharashtra', '2023-02-15'),
(103, 'Neha Singh', 'Bangalore', 'Karnataka', '2023-03-20'),
(104, 'Priya Verma', 'Pune', 'Maharashtra', '2023-04-25'),
(105, 'Karan Mehta', 'Hyderabad', 'Telangana', '2023-05-30'),
(106, 'Ankit Jain', 'Jaipur', 'Rajasthan', '2023-06-10'),
(107, 'Sneha Gupta', 'Lucknow', 'Uttar Pradesh', '2023-07-05');


In [0]:
SELECT * from practice.bronze.dim_customers;

## Step 8: Joins

In [0]:
-- BAsic INNER JOIN

SELECT 
  o.order_id,
  c.customer_name,
  c.city,
  o.product_id,
  o.quantity,
  o.price
FROM practice.bronze.orders_raw o
INNER JOIN practice.bronze.dim_customers c
ON o.customer_id = c.customer_id;

In [0]:
-- LEFT JOIN



SELECT 
  o.order_id,
  c.customer_name,
  o.price
FROM practice.bronze.orders_raw o
LEFT JOIN practice.bronze.dim_customers c
ON o.customer_id = c.customer_id;

In [0]:
-- TOTAL SPEND PER CUSTOMER (Join+AGG)

SELECT
  c.customer_name,
  SUM(o.quantity*o.price) as total_spent
FROM practice.bronze.orders_raw o
JOIN practice.bronze.dim_customers c
ON o.customer_id = c.customer_id
GROUP BY c.customer_name
ORDER BY total_spent DESC;

## Step 9: Conditional Logic

## Step 10: Date Functions

## Step 11: Window Functions

In [0]:
SELECT * from ecommerce_analytics.data_warehouse.fact_order_items;

In [0]:
SELECT *,
row_number()
over(partition by customer_id
order BY amount DESC) as rank
from ecommerce_analytics.data_warehouse.fact_order_items;

In [0]:
select * from(

SELECT *,
row_number()
over(partition by customer_id
order BY amount DESC) as rank
from ecommerce_analytics.data_warehouse.fact_order_items) t where rank = 1

In [0]:
SELECT *,
sum(amount)
over(partition by customer_id
) as total_sale
from ecommerce_analytics.data_warehouse.fact_order_items;

In [0]:
SELECT *,
LAG(amount)
over(partition by customer_id
order BY amount) as total_sale
from ecommerce_analytics.data_warehouse.fact_order_items;

## Step 12: CTE

## Step 13: Subqueries

## Step 14: Create Cleaned Table (Silver Layer)

## Step 15: Create Aggregated Table (Gold Layer)


## Step 16: Query Gold Table

## Step 17: Create View

## Step 18: Optimize Table (Delta Lake)

## Key Learnings

- SQL is the backbone of data engineering
- Window functions are heavily used in interviews
- Always think in layers:
    - Bronze → Raw
    - Silver → Cleaned
    - Gold → Aggregated
- Use CTEs for readability
- Optimize tables for performance in Databricks